# Рубежный контроль №1 по ТМО

**Студент:** Павел Смит  
**Группа:** ИУ5-64Б  
**Вариант:** 12  
**Задача:** №2  
**Датасет:** №4 (Heart Disease Dataset)

Источник задания: <https://github.com/ugapanyuk/courses_current/wiki/TMO_RK_1>


## Постановка задачи

Для варианта 12 требуется решить **задачу №2** на **датасете №4**.

Содержательно это означает:

1. Выбрать один категориальный и один количественный признак.
2. Проанализировать пропуски и обработать их.
3. Обосновать выбранные методы заполнения.
4. Построить обязательный `violin plot` для произвольной колонки данных.
5. Сделать вывод, какие признаки стоит использовать для дальнейшего обучения моделей.

В исходном датасете пропуски отсутствуют, поэтому по примечанию из задания они были **внесены искусственно**:

- для `chol`: 51 пропусков;
- для `thal`: 41 пропусков.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from matplotlib import pyplot as plt

sns.set_theme(style="whitegrid", context="talk")

ROOT = Path.cwd()
DATA_PATH = ROOT / "data" / "heart.csv"
IMG_DIR = ROOT / "output" / "img"
IMG_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
CHOL_MISSING_SHARE = 0.05
THAL_MISSING_SHARE = 0.04


In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Размер датасета: {df.shape[0]} строк, {df.shape[1]} столбцов")
display(df.head())
display(df.isna().sum().to_frame("missing_count").T)
display(df["target"].value_counts().rename(index={0: "Есть заболевание", 1: "Нет заболевания"}).to_frame("count").T)


## Выбор признаков и методов обработки

- **Количественный признак:** `chol`  
  Признак имеет положительную асимметрию (`skew = 1.074`), поэтому для него использована **медиана**.
- **Категориальный признак:** `thal`  
  Для него использовано заполнение **модой**, так как это категориальный код, и наиболее частое значение сохраняет исходное распределение лучше среднего или медианы.

После внесения пропусков использованы следующие значения:

- `chol` -> медиана = **240.0**
- `thal` -> мода = **2**


In [ ]:
rng = np.random.default_rng(SEED)
chol_missing_count = int(round(len(df) * CHOL_MISSING_SHARE))
thal_missing_count = int(round(len(df) * THAL_MISSING_SHARE))

chol_idx = rng.choice(df.index.to_numpy(), size=chol_missing_count, replace=False)
free_idx = np.setdiff1d(df.index.to_numpy(), chol_idx)
thal_idx = rng.choice(free_idx, size=thal_missing_count, replace=False)

df_missing = df.copy()
df_missing.loc[chol_idx, "chol"] = np.nan
df_missing.loc[thal_idx, "thal"] = np.nan

display(df_missing[["chol", "thal"]].isna().sum().to_frame("missing_count").T)
display(df_missing[["chol", "thal"]].describe())
print("Асимметрия chol:", round(df["chol"].skew(), 3))


In [ ]:
chol_median = float(df_missing["chol"].median())
thal_mode = int(df_missing["thal"].mode().iloc[0])

df_processed = df_missing.copy()
df_processed["chol"] = df_processed["chol"].fillna(chol_median)
df_processed["thal"] = df_processed["thal"].fillna(thal_mode).astype(int)
df_processed["target_label"] = df_processed["target"].map({0: "Есть заболевание", 1: "Нет заболевания"})

comparison = pd.DataFrame(
    {
        "До обработки": df_missing[["chol", "thal"]].isna().sum(),
        "После обработки": df_processed[["chol", "thal"]].isna().sum(),
    }
)
display(comparison)
print("Медиана chol:", chol_median)
print("Мода thal:", thal_mode)


In [ ]:
ax = comparison.plot(kind="bar", figsize=(8, 5), color=["#cc6677", "#4477aa"])
ax.set_title("Пропуски до и после обработки")
ax.set_xlabel("Признак")
ax.set_ylabel("Количество пропусков")
plt.tight_layout()
plt.savefig(IMG_DIR / "missing_comparison.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close()

plt.figure(figsize=(9, 5))
sns.violinplot(
    data=df_processed,
    x="target_label",
    hue="target_label",
    y="chol",
    inner="quartile",
    palette=["#cc6677", "#4477aa"],
    legend=False,
)
plt.title("Violin plot для признака chol по целевому классу")
plt.xlabel("Класс")
plt.ylabel("Уровень холестерина")
plt.tight_layout()
plt.savefig(IMG_DIR / "chol_violin.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close()

plt.figure(figsize=(9, 5))
sns.countplot(
    data=df_processed,
    x="thal",
    hue="target_label",
    palette=["#cc6677", "#4477aa"],
)
plt.title("Распределение thal после заполнения пропусков")
plt.xlabel("Категория thal")
plt.ylabel("Количество наблюдений")
plt.tight_layout()
plt.savefig(IMG_DIR / "thal_countplot.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close()


## Корреляция признаков с target

Наиболее выраженные по модулю связи после обработки пропусков:

- `oldpeak`: `-0.438`
- `exang`: `-0.438`
- `cp`: `0.435`
- `thalach`: `0.423`
- `ca`: `-0.382`
- `slope`: `0.346`
- `thal`: `-0.342`
- `sex`: `-0.280`

In [ ]:
corr_with_target = (
    df_processed.drop(columns=["target_label"])
    .corr(numeric_only=True)["target"]
    .drop("target")
    .sort_values(key=lambda s: s.abs(), ascending=False)
)
display(corr_with_target.to_frame("corr_with_target"))
print("Рекомендуемые признаки для приоритетного использования:")
print(["oldpeak", "exang", "cp", "thalach", "ca", "slope", "thal", "sex", "age"])
print("Менее приоритетные признаки по одиночной связи с target:")
print(["fbs", "chol", "restecg", "trestbps"])


## Выводы

1. Для `chol` медиана предпочтительнее среднего, потому что распределение имеет правый хвост и выбросы.
2. Для `thal` корректно использовать моду, так как это категориальный признак, записанный числами.
3. Для дальнейшего ML разумно оставить все медицински осмысленные признаки, но при отборе в первую очередь опираться на:
   `oldpeak, exang, cp, thalach, ca, slope, thal, sex, age`
4. Признаки с наименее выраженной индивидуальной связью с целевой переменной в этом датасете:
   `fbs, chol, restecg, trestbps`
